# Ruling out the fixable causes

Notebook 03 ended on a surprise: settling the double tightest of anything still trailed linear. So
the agreement cap is not the oscillation. This notebook eliminates the remaining *training-side*
suspects one by one — reward variance, training budget, and coverage — and shows the residual is the
same shape in every case: a one-directional **over-doubling bias** that none of them moves. That
leaves representation (notebook 05) as the only thing left standing.

In [ ]:
import sys, json; sys.path.insert(0, '.'); sys.path.insert(0, '..')
import random, math, statistics as st
from pathlib import Path
import matplotlib.pyplot as plt
from blackjack_rl.analysis_loader import load_runs

df = load_runs(); dqn = df[df.method == 'dqn']

def diff_cells(path):
    return json.load(open(Path(path) / 'record.json', encoding='utf-8'))['diff']['cells']

def breakdown(path):
    cells = diff_cells(path)
    dis = [c for c in cells if c['category'] == 'genuine_disagreement']
    over = [c for c in dis if c['agent_action'] == 'double' and c['basic_action'] != 'double']
    under = [c for c in dis if c['basic_action'] == 'double' and c['agent_action'] != 'double']
    near = sum(1 for c in cells if c['category'] == 'near_equal_ev')
    gap = st.mean(abs(c['agent_q'] - c['basic_q']) for c in dis) if dis else 0.0
    return dict(genuine=len(dis), over=len(over), under=len(under), near=near, self_gap=gap)

# the residual to explain — take the best-settled long run (5M flat-then-decay)
final = dqn[(dqn.lr_hold_until > 0) & (dqn.episodes == 5_000_000)].iloc[0]
b = breakdown(final.path)
print(f'residual disagreements ({final.run}): {b["genuine"]} genuine')
print(f'   over-double (doubles when it should not): {b["over"]}')
print(f'   under-double (misses a real double)     : {b["under"]}')
print('=> the entire residual is over-doubling, one direction. that is the thing to explain.')

## Suspect 1 — reward variance (a control variate)

Most of a hand's ±2 swing is the **dealer's** outcome, shared across hit/stand/double and so noise
w.r.t. *your* decision. A control variate subtracts a mean-zero, action-independent dealer baseline —
EV preserved, policy preserved. It's correct and unbiased. But it can only cancel the variance from
the source it targets, and for the **double** the dealer is a minority of the variance — the bulk is
the player's own one-card draw, which is intrinsic. So it leaves the double untouched.

In [ ]:
V = list(range(2, 11)) + [10, 10, 10] + [11]
def hv(cards):
    t, a = sum(cards), cards.count(11)
    while t > 21 and a: t -= 10; a -= 1
    return t
def dealer(up, rng):
    c = [up, rng.choice(V)]
    while hv(c) < 17: c.append(rng.choice(V))
    t = hv(c); return 0 if t > 21 else t
def rew(pf, d): return -1.0 if pf > 21 else (1.0 if d == 0 else (1.0 if pf > d else (-1.0 if pf < d else 0.0)))
def decompose(start, up, n=40_000):
    rng = random.Random(0); rows = []
    for _ in range(n):
        pf = hv(start + [rng.choice(V)]); d = dealer(up, rng); rows.append((2 * rew(pf, d), pf, d))
    r = [x[0] for x in rows]; var = st.pvariance(r); mu = st.mean(r)
    def expl(k):
        g = {}
        for x in rows: g.setdefault(k(x), []).append(x[0])
        return sum(len(v) * (st.mean(v) - mu) ** 2 for v in g.values()) / len(rows)
    return var, expl(lambda x: x[2]) / var, expl(lambda x: x[1]) / var
print(f"{'double cell':18}{'Var':>6}{'dealer':>9}{'player-draw':>13}")
for name, cards, up in [('soft-20 v8', [11, 9], 8), ('hard-11 v6', [5, 6], 6), ('hard-16 v10', [10, 6], 10)]:
    var, fd, fp = decompose(cards, up)
    print(f'{name:18}{var:>6.2f}{fd:>8.0%}{fp:>12.0%}')
rb = dqn[dqn.reward_baseline != 'none']
vanilla = dqn[(dqn.reward_baseline == 'none') & (dqn.lr_schedule == 'constant') & dqn.soft20_double_std.notna()]
print(f'\nsoft-20 double-std: with baseline {rb.soft20_double_std.dropna().median():.2f}  vs  none {vanilla.soft20_double_std.median():.2f}  (inside the same band -> inert)')

## Suspect 2 — training budget

Maybe the double just needs more learning. Measuring "total learning" as the area under the lr curve
(sum of lr over episodes), the flat-then-decay 1.5M run gave the *double* a tiny budget — masked in
phase 1, then fed the harmonic's thin tail. The 5M run hands it ~4.5× more. If budget were the cap,
agreement should jump. It doesn't.

In [ ]:
def harm_mean(s, e): c = s / e - 1; return (s / c) * math.log(1 + c)
def double_budget(total, hold, s, e, sched):
    span = total - hold
    return span * (s / 2 if sched == 'linear' else harm_mean(s, e))   # double only learns after the switch
print('double-only learning budget (sum of lr the double action sees):')
print(f"  linear 1e-3->0, 1.5M (no curriculum) : {double_budget(1_500_000, 0, 1e-3, 0, 'linear'):7.0f}")
print(f"  flat-then-decay 1.5M                 : {double_budget(1_500_000, 500_000, 1e-3, 1e-5, 'harm'):7.0f}")
print(f"  flat-then-decay 5M                   : {double_budget(5_000_000, 500_000, 1e-3, 1e-5, 'harm'):7.0f}")
ftd15 = dqn[(dqn.lr_hold_until > 0) & (dqn.episodes == 1_500_000)].iloc[0]
ftd5m = dqn[(dqn.lr_hold_until > 0) & (dqn.episodes == 5_000_000)].iloc[0]
print(f'\n  1.5M flat-then-decay: agree {ftd15.agreement:.1%}  edge {ftd15.edge_pct:.2f}%  over-dbl {breakdown(ftd15.path)["over"]}')
print(f'  5M   flat-then-decay: agree {ftd5m.agreement:.1%}  edge {ftd5m.edge_pct:.2f}%  over-dbl {breakdown(ftd5m.path)["over"]}')
print('  -> 4.5x the budget, agreement and over-doubling essentially unchanged. budget is not the cap.')

## Suspect 3 — coverage, and a warning about metrics

Two related points. First, the seed-to-seed scatter in agreement is a *rare-cell starvation* effect
(once the greedy policy commits and epsilon anneals, rare cells stop getting fresh data and drift) —
real, but it produces seed *noise*, not the systematic over-double. Second, a metric trap. From 1.5M
to 5M the agent's values converged tighter, so several borderline cells slipped under the EV-tolerance
and were **reclassified from "genuine" disagreement to "near-equal-EV"** — the diff *looks* cleaner.
But the true **edge did not improve**. A softer-looking disagreement breakdown is the agent grading
its own confidence; only edge (and agreement) say whether the policy actually got better.

In [ ]:
print(f"{'run':22}{'agree':>7}{'edge':>7}{'genuine':>9}{'near':>6}")
for lab, g in [('1.5M flat-then-decay', ftd15), ('5M flat-then-decay', ftd5m)]:
    bd = breakdown(g.path)
    print(f'{lab:22}{g.agreement:>7.1%}{g.edge_pct:>6.2f}%{bd["genuine"]:>9}{bd["near"]:>6}')
print('\ngenuine fell and near rose (values converged) — yet edge is flat. the classification softened, the policy did not.')

## Conclusion

Three training-side fixes, three nulls: a control variate can't quiet the double's *intrinsic* draw
variance; 4.5× more budget doesn't lift agreement; and coverage explains only seed noise. Through all
of it the residual keeps the same fingerprint — **over-doubling, one direction, concentrated in a
handful of boundary cells**. That is the signature of a *bias*, not variance or under-training, and a
bias in specific cells is a property of how the function approximator *represents* the decision map.
Notebook 05 looks directly at that representation and finds the errors exactly where it predicts.